In [2]:
import pickle
with open("./docs/FWG_with_global.pkl", "rb") as f:
    docs = pickle.load(f)
len(docs)

7

In [14]:
print(docs[2].page_content)

Global Metadata: {"title": "TECHNICAL SPECIFICATION FOR F.W. GENERATOR", "ship_numbers": ["8250", "8251"], "product_name": "F.W. GENERATOR", "specifications": "Low-pressure evaporating type (M/E jacket water heating), Shell & Tube, 25 ton/day capacity with 15% fouling margin, max 10 PPM salinity, M/E jacket cooling F.W. heating medium, S.W. cooling medium", "document_type": "Technical Specification"} 

 Global Context: This chunk is the opening technical-specification sheet (sheet 1/5) that immediately follows the package list and begins the detailed performance, construction and material requirements for the F.W. generator. 

 Content: | SHIP NO.   | TECHNICAL SPECIFICATION FOR   | SHEET PAGE   |
|------------|-------------------------------|--------------|
| 8250/8251  | F.W. GENERATOR                | 1/5          |

1. Type

Low pressure evaporating type. (M/E jacket water heating) Shell &amp; Tube type

## 2. Particular

- 1) Q'ty/ship

: One(1) Set

- 2) Capacity                 

In [6]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

def build_passages(doc, chunk_size=5):
    """
    Split a document into fixed-size sentence chunks (passages).

    This function tokenizes the input document into sentences and groups them
    into passages of a specified size. Each passage contains a list of sentences
    and a unique passage ID.

    Args:
        doc (str): Input text document to be split into passages.
        chunk_size (int, optional): Number of sentences per passage.
            Defaults to 5.

    Returns:
        List[dict]: A list of passages, where each passage is a dictionary with:
            - "passage_id" (str): Unique identifier for the passage (e.g., "P0", "P5", ...).
            - "sentences" (List[str]): List of sentences in the passage.

    Example:
        >>> doc = "Sentence one. Sentence two. Sentence three. Sentence four."
        >>> build_passages(doc, chunk_size=2)
        [
            {"passage_id": "P0", "sentences": ["Sentence one.", "Sentence two."]},
            {"passage_id": "P2", "sentences": ["Sentence three.", "Sentence four."]}
        ]

    Notes:
        - Sentence tokenization depends on the behavior of `sent_tokenize`.
        - The last passage may contain fewer sentences if the total number
          of sentences is not divisible by `chunk_size`.
    """
    sentences = sent_tokenize(doc.page_content)
    print(f">>> cnt of sentences: {len(sentences)}")
    passages = []

    for i in range(0, len(sentences), chunk_size):
        chunk = sentences[i:i+chunk_size]
        passages.append({
            "passage_id": f"P{i}",
            "sentences": chunk
        })
    return passages

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jongb\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [8]:
passages = build_passages(doc=docs[2], chunk_size=300)
passages

>>> cnt of sentences: 27


[{'passage_id': 'P0',
  'sentences': ['Global Metadata: {"title": "TECHNICAL SPECIFICATION FOR F.W.',
   'GENERATOR", "ship_numbers": ["8250", "8251"], "product_name": "F.W.',
   'GENERATOR", "specifications": "Low-pressure evaporating type (M/E jacket water heating), Shell & Tube, 25 ton/day capacity with 15% fouling margin, max 10 PPM salinity, M/E jacket cooling F.W.',
   'heating medium, S.W.',
   'cooling medium", "document_type": "Technical Specification"} \n\n Global Context: This chunk is the opening technical-specification sheet (sheet 1/5) that immediately follows the package list and begins the detailed performance, construction and material requirements for the F.W.',
   'generator.',
   'Content: | SHIP NO.',
   '| TECHNICAL SPECIFICATION FOR   | SHEET PAGE   |\n|------------|-------------------------------|--------------|\n| 8250/8251  | F.W.',
   'GENERATOR                | 1/5          |\n\n1.',
   'Type\n\nLow pressure evaporating type.',
   '(M/E jacket water heating)

In [10]:
import spacy

nlp = spacy.load("../model/en_core_web_sm")

def extract_entities(text):
    doc = nlp(text)
    return list(set([ent.text for ent in doc.ents]))
    # return list(set([ent.text for ent in doc if ent.pos_=="NOUN"]))  # 명사 추출 방식

d:\auto_vectordb\.venv\Lib\site-packages\spacy\util.py:971: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [15]:
text = docs[2].page_content
ner = extract_entities(text=text)
ner

['## 2',
 'Source           ',
 'HYUNDAI',
 '1',
 'max',
 'F.W. GENERATOR                |',
 'S.W.                       ',
 'Shell',
 'Tube',
 '25 ton',
 'Capacity         ',
 '8250/8251',
 '## 1) Construction',
 'Maker',
 'Shell & Tube',
 '15%',
 'PPM',
 '32                         ',
 'Distillate',
 'F.W.',
 'Max',
 '83',
 'COOLING MEDIUM             ',
 'Shaft',
 'Global Metadata',
 'm3/h',
 '8250',
 '4.5',
 'Technical Specification',
 'Shell &',
 '## 3',
 'Inlet',
 '10',
 'SHIP NO',
 'evaporating chamber',
 '1/5',
 '01-25 PM']

In [16]:
## Domain NER 커스터마이징
import spacy

nlp = spacy.load("../model/en_core_web_sm")

ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = [
    {"label": "PRODUCT", "pattern": "GPT-4"},
    {"label": "PRODUCT", "pattern": "GRAPH"},
    {"label": "ERROR", "pattern": [{"TEXT": {"REGEX": "ERR[0-9]+"}}]},
    {"label": "API", "pattern": "OpenAI API"}
]

ruler.add_patterns(patterns)
doc = nlp(docs[2].page_content)
[(ent.text, ent.label_) for ent in doc.ents]

d:\auto_vectordb\.venv\Lib\site-packages\spacy\util.py:971: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


[('Global Metadata', 'PERSON'),
 ('8250', 'CARDINAL'),
 ('Shell & Tube', 'ORG'),
 ('25 ton', 'QUANTITY'),
 ('15%', 'PERCENT'),
 ('max', 'PERSON'),
 ('10', 'CARDINAL'),
 ('PPM', 'ORG'),
 ('F.W.', 'ORG'),
 ('Technical Specification', 'WORK_OF_ART'),
 ('1/5', 'CARDINAL'),
 ('SHIP NO', 'ORG'),
 ('8250/8251', 'DATE'),
 ('F.W. GENERATOR                |', 'PERSON'),
 ('1/5', 'CARDINAL'),
 ('1', 'CARDINAL'),
 ('Shell &', 'ORG'),
 ('Tube', 'ORG'),
 ('## 2', 'MONEY'),
 ('25 ton', 'QUANTITY'),
 ('15%', 'PERCENT'),
 ('Shell', 'ORG'),
 ('Max', 'PERSON'),
 ('10', 'CARDINAL'),
 ('PPM', 'ORG'),
 ('COOLING MEDIUM             ', 'PERSON'),
 ('Source           ', 'PERSON'),
 ('HYUNDAI', 'ORG'),
 ('S.W.                       ', 'PERSON'),
 ('m3/h', 'ORG'),
 ('Maker', 'ORG'),
 ('Maker', 'ORG'),
 ('Inlet', 'LOC'),
 ('83', 'CARDINAL'),
 ('32                         ', 'QUANTITY'),
 ('4.5', 'CARDINAL'),
 ('Maker', 'ORG'),
 ('Distillate', 'ORG'),
 ('Capacity         ', 'ORG'),
 ('Maker', 'ORG'),
 ('Maker', 'O

In [74]:
doc = nlp(text.upper())
[(ent.text, ent.label_) for ent in doc.ents]

[('ANALYSIS FOR', 'WORK_OF_ART'), ('GRAPH', 'PRODUCT')]

In [11]:
def build_graph(passages):
    sentence_nodes = []
    entity_map = {}

    for p in passages:
        for i, sent in enumerate(p["sentences"]):
            sid = f"S_{i}"
            entities = extract_entities(sent)

            sentence_nodes.append({
                "sentence_id": sid,
                "text": sent,
                "entities": entities,
                "passage_id": p["passage_id"]
            })

            for e in entities:
                entity_map.setdefault(e, {
                    "entity": e,
                    "sentences": [],
                    "passages": set()
                })
                entity_map[e]["sentences"].append(sid)
                entity_map[e]["passages"].add(p["passage_id"])

    return sentence_nodes, list(entity_map.values())

In [12]:
res = build_graph(passages=passages)
res

([{'sentence_id': 'S_0',
   'text': 'Global Metadata: {"title": "TECHNICAL SPECIFICATION FOR F.W.',
   'entities': ['Global Metadata'],
   'passage_id': 'P0'},
  {'sentence_id': 'S_1',
   'text': 'GENERATOR", "ship_numbers": ["8250", "8251"], "product_name": "F.W.',
   'entities': ['8250'],
   'passage_id': 'P0'},
  {'sentence_id': 'S_2',
   'text': 'GENERATOR", "specifications": "Low-pressure evaporating type (M/E jacket water heating), Shell & Tube, 25 ton/day capacity with 15% fouling margin, max 10 PPM salinity, M/E jacket cooling F.W.',
   'entities': ['Shell & Tube', '15%', 'PPM', 'max', 'F.W.', '10', '25 ton'],
   'passage_id': 'P0'},
  {'sentence_id': 'S_3',
   'text': 'heating medium, S.W.',
   'entities': [],
   'passage_id': 'P0'},
  {'sentence_id': 'S_4',
   'text': 'cooling medium", "document_type": "Technical Specification"} \n\n Global Context: This chunk is the opening technical-specification sheet (sheet 1/5) that immediately follows the package list and begins the det